# A/B Testing with Amazon Bedrock AgentCore

This notebook demonstrates **target-based A/B testing** using Amazon Bedrock AgentCore.

All cells use `%%bash` — no Python required.

**Prerequisites:** Python 3.12+, `uv`, Node.js, AWS CLI >= 2.34, CDK bootstrapped, Bedrock model access.

**Run this notebook from the `ab_testing/` directory.**

## 1. Check Prerequisites

In [ ]:
%%bash
scripts/check_prerequisites.sh

## 2. Package Agents

In [ ]:
%%bash
target_based_variants/scripts/package_agents.sh target_based_variants/agents

## 3. Deploy Runtimes + Evaluation Configs

Deploys `fixFirstAgent-ABTestingStack`: two AgentCore Runtimes, two Online Evaluation Configs, IAM roles, SSM parameters.

In [ ]:
%%bash
cd target_based_variants/cdk_ab_testing && npx cdk deploy fixFirstAgent-ABTestingStack --require-approval never

## 4. Wait for Runtimes to Become READY

In [ ]:
%%bash
REGION=$(aws configure get region 2>/dev/null)
REGION=${REGION:-us-east-1}
CONTROL_ARN=$(aws ssm get-parameter --name /fixFirstAgent/control-runtime-arn --query Parameter.Value --output text --region $REGION)
REFINED_ARN=$(aws ssm get-parameter --name /fixFirstAgent/refined-runtime-arn --query Parameter.Value --output text --region $REGION)
CONTROL_ID=$(echo "$CONTROL_ARN" | awk -F/ '{print $NF}')
REFINED_ID=$(echo "$REFINED_ARN" | awk -F/ '{print $NF}')

for NAME_ID in "Control:$CONTROL_ID" "Treatment:$REFINED_ID"; do
    NAME=${NAME_ID%%:*}
    RID=${NAME_ID#*:}
    echo "Waiting for $NAME runtime ($RID)..."
    for i in $(seq 1 30); do
        STATUS=$(aws bedrock-agentcore-control get-agent-runtime --agent-runtime-id "$RID" --region $REGION --query status --output text 2>/dev/null)
        if [ "$STATUS" = "READY" ]; then
            echo "  $NAME is READY"
            break
        fi
        echo "  [$i/30] $STATUS"
        sleep 20
    done
    if [ "$STATUS" != "READY" ]; then
        echo "ERROR: $NAME not READY after 10 minutes"
        exit 1
    fi
done
echo "Both runtimes READY."

## 5. Send Requests to Each Agent (Generate Evaluation Data)

Sends 10 prompts to each agent directly via `invoke-agent-runtime` to generate evaluation scores.

In [ ]:
%%bash
REGION=$(aws configure get region 2>/dev/null)
REGION=${REGION:-us-east-1}
CONTROL_ARN=$(aws ssm get-parameter --name /fixFirstAgent/control-runtime-arn --query Parameter.Value --output text --region $REGION)
REFINED_ARN=$(aws ssm get-parameter --name /fixFirstAgent/refined-runtime-arn --query Parameter.Value --output text --region $REGION)

for VARIANT in "Control:$CONTROL_ARN" "Treatment:$REFINED_ARN"; do
    NAME=${VARIANT%%:*}
    ARN=${VARIANT#*:}
    echo "=== $NAME Agent ==="
    COUNT=0
    while IFS= read -r prompt && [ $COUNT -lt 10 ]; do
        [ -z "$prompt" ] && continue
        COUNT=$((COUNT + 1))
        echo "{\"prompt\": \"$prompt\"}" > _payload.json
        aws bedrock-agentcore invoke-agent-runtime \
            --agent-runtime-arn "$ARN" \
            --qualifier DEFAULT \
            --runtime-session-id "eval-${NAME,,}-$COUNT" \
            --payload fileb://_payload.json \
            --region $REGION _response.json >/dev/null 2>&1
        echo "  [$COUNT/10] $prompt"
    done < prompts.txt
    echo "$NAME: $COUNT requests sent"
    echo ""
done

rm -f _payload.json _response.json
echo "All requests sent. Evaluation scores will appear in ~15 minutes."

## 6. Deploy Gateway + Targets + A/B Test

Deploys `fixFirstAgent-ABGatewayStack` which creates the gateway, HTTP targets, and 50/50 A/B test via `ILocalBundling`.

In [ ]:
%%bash
REGION=$(aws configure get region 2>/dev/null)
REGION=${REGION:-us-east-1}
CONTROL_RUNTIME_ARN=$(aws ssm get-parameter --name /fixFirstAgent/control-runtime-arn --query Parameter.Value --output text --region $REGION)
REFINED_RUNTIME_ARN=$(aws ssm get-parameter --name /fixFirstAgent/refined-runtime-arn --query Parameter.Value --output text --region $REGION)
CONTROL_EVAL_ARN=$(aws ssm get-parameter --name /fixFirstAgent/control-eval-arn --query Parameter.Value --output text --region $REGION)
TREATMENT_EVAL_ARN=$(aws ssm get-parameter --name /fixFirstAgent/treatment-eval-arn --query Parameter.Value --output text --region $REGION)

cd target_based_variants/cdk_ab_gateway && npx cdk deploy fixFirstAgent-ABGatewayStack --require-approval never \
    -c "controlRuntimeArn=$CONTROL_RUNTIME_ARN" \
    -c "refinedRuntimeArn=$REFINED_RUNTIME_ARN" \
    -c "controlEvalArn=$CONTROL_EVAL_ARN" \
    -c "treatmentEvalArn=$TREATMENT_EVAL_ARN"

## 7. Send Traffic Through Gateway

Sends 20 prompts through the gateway. The A/B test splits traffic 50/50 between control and treatment.

In [ ]:
%%bash
REGION=$(aws configure get region 2>/dev/null)
REGION=${REGION:-us-east-1}
GATEWAY_URL=$(aws ssm get-parameter --name /fixFirstAgent/ab-gateway-url --query Parameter.Value --output text --region $REGION)
scripts/send_traffic.sh "$GATEWAY_URL" "$REGION" prompts.txt

## 8. Check A/B Test Results

Re-run this cell after ~15 minutes to see statistical results.

- `p-value < 0.05` + positive change = treatment is significantly better
- `p-value >= 0.05` = not enough evidence yet

In [ ]:
%%bash
REGION=$(aws configure get region 2>/dev/null)
REGION=${REGION:-us-east-1}
AB_TEST_ID=$(aws ssm get-parameter --name /fixFirstAgent/ab-test-id --query Parameter.Value --output text --region $REGION)
echo "A/B Test ID: $AB_TEST_ID"
echo ""
aws bedrock-agentcore get-ab-test --ab-test-id "$AB_TEST_ID" --region $REGION \
    --query "{Status:status,Execution:executionStatus,AnalysisTime:results.analysisTimestamp,Control:{Mean:results.evaluatorMetrics[0].controlStats.mean,Samples:results.evaluatorMetrics[0].controlStats.sampleSize},Treatment:{Mean:results.evaluatorMetrics[0].variantResults[0].mean,Samples:results.evaluatorMetrics[0].variantResults[0].sampleSize,PercentChange:results.evaluatorMetrics[0].variantResults[0].percentChange,PValue:results.evaluatorMetrics[0].variantResults[0].pValue,Significant:results.evaluatorMetrics[0].variantResults[0].isSignificant}}" \
    --output table

## 9. Stop the A/B Test

In [ ]:
%%bash
REGION=$(aws configure get region 2>/dev/null)
REGION=${REGION:-us-east-1}
AB_TEST_ID=$(aws ssm get-parameter --name /fixFirstAgent/ab-test-id --query Parameter.Value --output text --region $REGION)
aws bedrock-agentcore update-ab-test --ab-test-id "$AB_TEST_ID" --execution-status STOPPED --region $REGION
echo "A/B test $AB_TEST_ID stopped."

## 10. Cleanup

Removes all resources: A/B test, gateway, targets, IAM role, CDK stacks.

In [ ]:
%%bash
REGION=$(aws configure get region 2>/dev/null)
export APP_NAME=fixFirstAgent
export AWS_REGION=${REGION:-us-east-1}
target_based_variants/scripts/cleanup_all.sh